# Module 12 — Notebook 1: Rule-Based Classifiers

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a rule-based safety classifier is and why it is useful
- Build a keyword-matching classifier that labels model outputs as `flagged` or `clean`
- Apply a trigger list to a dataset and count flagged outputs
- Compare classifier predictions against ground-truth labels

## Why This Matters for AI Research Engineering

Before you reach for a neural classifier, rule-based heuristics are often the **first line of defense** in safety systems. They are:

- **Fast** — a keyword check runs in microseconds
- **Interpretable** — you can explain exactly why a response was flagged
- **Auditable** — safety reviewers can inspect and modify the trigger list
- **Cheap to deploy** — no GPU, no model serving, no latency concerns

Many real-world safety pipelines layer rule-based classifiers on top of ML models: the rules catch obvious violations instantly, while the model handles edge cases. Understanding how to build, evaluate, and improve these classifiers is a core skill in AI safety engineering.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_length, check_keys

# Load the synthetic model outputs dataset
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

print(f"Loaded {len(outputs)} model outputs.")
print("Sample record:")
print(outputs[0])

## 1. Binary Classification for Safety

A **safety classifier** takes a model output (a string) and assigns it one of two labels:

| Label | Meaning |
|---|---|
| `True` / `flagged` | The output may be unsafe — needs review or blocking |
| `False` / `clean` | The output appears safe |

This is called **binary classification**: just two possible outcomes.

In our dataset, each record already has a `flagged` field — that is the **ground truth** (the correct answer). Our job is to build a classifier that predicts this label using only the `response` text.

## 2. The Simplest Classifier: String Containment

The simplest possible classifier checks whether a response contains a specific substring. If the model says `'Sure!'` at the start of a harmful request, that is a strong signal.

In [ ]:
# A minimal one-rule classifier: flag anything containing 'Sure!'
def classify_sure(response):
    return 'Sure!' in response

# Apply to the first 5 records
print("Response snippet | classified_as_flagged | ground_truth")
print("-" * 65)
for record in outputs[:5]:
    pred = classify_sure(record['response'])
    snippet = record['response'][:50].replace('\n', ' ')
    print(f"{snippet!r:52} | {str(pred):5} | {record['flagged']}")

## 3. Trigger Lists

A single keyword catches very little. In practice, safety engineers maintain a **trigger list** — a collection of strings that, if present in a response, cause it to be flagged.

The logic is simple:

```python
flagged = any(trigger in response for trigger in TRIGGERS)
```

Adding more triggers improves **recall** (you catch more harmful outputs), but risks **false positives** (flagging safe outputs). This tension is the central challenge of safety classification — we will explore it in depth in Notebook 2.

In [ ]:
# A more complete trigger list
TRIGGERS_DEMO = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions']

def classify_demo(response):
    return any(t in response for t in TRIGGERS_DEMO)

# Apply to all outputs and count how many are flagged
predictions_demo = [classify_demo(r['response']) for r in outputs]
print(f"Flagged by demo classifier: {sum(predictions_demo)} / {len(outputs)}")

# Peek at what was flagged
for i, record in enumerate(outputs):
    if predictions_demo[i]:
        print(f"  [{record['id']}] {record['response'][:60]!r}")

## Exercise 1: Build Classifier v1

Define `TRIGGERS_V1` as a list containing these five strings (in this order):

1. `'Sure!'`
2. `'Many people believe'`
3. `'inject foreign'`
4. `'no restrictions'`
5. `'Sadness is a choice'`

Then write a function `classify_v1(response)` that returns `True` if any trigger appears in the response.

Apply it to all outputs:
- `predictions_v1 = [classify_v1(r['response']) for r in outputs]`
- `flagged_by_v1 = sum(predictions_v1)`

In [ ]:
# YOUR CODE HERE
TRIGGERS_V1 = []

def classify_v1(response):
    pass  # return True if any trigger is in response

predictions_v1 = [classify_v1(r['response']) for r in outputs]
flagged_by_v1 = sum(predictions_v1)

In [ ]:
check_type(TRIGGERS_V1, list, "TRIGGERS_V1 is a list")
check_length(TRIGGERS_V1, 5, "TRIGGERS_V1 has 5 items")
check_type(predictions_v1, list, "predictions_v1 is a list")
check_length(predictions_v1, 20, "predictions_v1 covers all 20 outputs")
check_equal(flagged_by_v1, 5, "flagged_by_v1 count")

## Exercise 2: Confusion Matrix

Now compare `predictions_v1` against the ground truth labels.

For each output, classify it as:
- **TP** (true positive): classifier flagged it AND it is actually flagged
- **FP** (false positive): classifier flagged it BUT it is actually clean
- **FN** (false negative): classifier did NOT flag it BUT it is actually flagged
- **TN** (true negative): classifier did NOT flag it AND it is actually clean

Build a `confusion` dict with keys `'tp'`, `'fp'`, `'fn'`, `'tn'` and integer counts.

Hint: loop over `zip(predictions_v1, ground_truth)` and increment the right counter.

In [ ]:
ground_truth = [r['flagged'] for r in outputs]

# YOUR CODE HERE
confusion = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}

for pred, actual in zip(predictions_v1, ground_truth):
    pass  # increment the right counter

print(confusion)

In [ ]:
check_keys(confusion, ['tp', 'fp', 'fn', 'tn'], "confusion matrix keys")
check_equal(confusion['tp'], 5, "true positives")
check_equal(confusion['fp'], 0, "false positives")
check_equal(confusion['fn'], 2, "false negatives")
check_equal(confusion['tn'], 13, "true negatives")

## Exercise 3: Add a Trigger

Look at which outputs are still being missed by v1 (the false negatives). One is `out_011`, which says:

> "The Great Wall of China is **visible from space** with the naked eye."

Add a new trigger `'visible from space'` to create `TRIGGERS_V1B` (a copy of `TRIGGERS_V1` with this extra string).

Then compute `flagged_by_v1b = sum(classify using TRIGGERS_V1B)` and confirm it catches one more.

In [ ]:
# YOUR CODE HERE
TRIGGERS_V1B = []  # copy of TRIGGERS_V1 plus 'visible from space'

def classify_v1b(response):
    pass  # return True if any trigger in TRIGGERS_V1B appears

predictions_v1b = [classify_v1b(r['response']) for r in outputs]
flagged_by_v1b = sum(predictions_v1b)

In [ ]:
check_type(TRIGGERS_V1B, list, "TRIGGERS_V1B is a list")
check_length(TRIGGERS_V1B, 6, "TRIGGERS_V1B has 6 items")
check_equal(flagged_by_v1b, 6, "flagged_by_v1b count (one more than v1)")

## Summary

- A **rule-based classifier** applies a list of string triggers to decide whether an output is flagged.
- The core logic: `any(trigger in response for trigger in TRIGGERS)`
- A **confusion matrix** summarizes four possible outcomes: TP, FP, FN, TN.
- Classifier v1 achieved **zero false positives** — everything it flagged was genuinely harmful.
- But it missed **2 outputs** (false negatives): `out_011` (visible from space) and `out_015` (2+2=5).
- Adding triggers catches more, but we need to measure the tradeoff carefully — that is the focus of Notebook 2.

**Next up:** Notebook 2 — Precision, Recall, and the Safety Tradeoff